## **Setup**

In [86]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [87]:
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parents[0]  # notebooks/ is one level down
sys.path.append(str(PROJECT_ROOT / "src"))

In [106]:
from chatGnT.config import CFG, ensure_dirs
import chatGnT.utils as utils
import chatGnT.data.load as load
import chatGnT.data.preprocess as preprocess
import chatGnT.data.tokenize as tokenize
ensure_dirs(CFG)
import time
import math
import torch
from torch.nn.utils.rnn import pad_sequence
import torch.nn as nn  # for embedding layer
from torch.utils.data import TensorDataset, DataLoader
import chatGnT.models.transformer as transformer
import chatGnT.models.positional_encoding as positional_encoding
import chatGnT.models.train as train
import chatGnT.models.evaluate as evaluate
import chatGnT.models.predict as predict

## **Load & Clean Data**

In [89]:
data = load.load_all()
df = data["ingred"]
#TODO: revisit and handle things like "Twist of  Orange peel" "Top with..."
#TODO: how make units consistent - good opportunity for test functions?
df_clean = preprocess.clean_ingred(df)
# df_clean.head()


## **Make Vocab & Tokenize**

In [90]:
tokens = tokenize.recipe_to_tokens(df_clean)
print(tokens[0])

['<amt>1</amt>', '<unit>oz</unit>', '<ingred>Coconut rum</ingred>', '<sep>', '<amt>1/2</amt>', '<unit>oz</unit>', '<ingred>Amaretto</ingred>', '<sep>', '<amt>4</amt>', '<unit>oz</unit>', '<ingred>Orange juice</ingred>', '<sep>', '<amt>1/2</amt>', '<unit>oz</unit>', '<ingred>Grenadine</ingred>', '<sep>']


In [91]:
vocab = tokenize.make_vocab(tokens)
print(len(vocab))  # 611 classes to predict
inv_vocab = tokenize.invert_vocab(vocab)
tokens_padded = tokenize.embed_tokens(tokens, vocab)

611


In [92]:
# tokens_tensor = [torch.tensor(r) for r in tokens_padded]
tokens_tensor = torch.tensor(tokens_padded, dtype=torch.long)
print(tokens_tensor.shape)  # torch.Size([621, 48])

# Need to have [seq_len, batch_size] for TransformerEncoder
# tokens_tensor = tokens_tensor.transpose(0, 1)
# print(tokens_tensor.shape)  # torch.Size([49, 621])

torch.Size([621, 49])


In [93]:
print(len(tokens_padded))
type(tokens_padded)
print(tokens_padded[0])

621
[25, 578, 150, 509, 10, 578, 69, 509, 43, 578, 293, 509, 10, 578, 219, 509, 610, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [94]:
#TODO: need to do this elsewhere so matches batch size?


# TODO: for now, add padding mask but not a causal mask that block seeing future
pad_id = vocab["<pad>"]   # should be 0
# src_key_padding_mask shape = [batch_size, seq_len]
pad_mask = (tokens_tensor == pad_id).transpose(0, 1)
print(pad_mask.shape)   # [batch_size, seq_len]


torch.Size([49, 621])


## **Get Batches**

In [95]:
# Not shifting targets since want model to attend to all tokens
inputs = tokens_tensor
targets = tokens_tensor

dataset = TensorDataset(inputs, targets)
dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

print(dataloader.batch_size)  # 32
print(len(dataloader))  # 20 = 621 / 32

32
20


## **Train Model**

In [96]:
embed_size=16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = transformer.TransformerModel(
    ntoken=len(vocab),
    ninp=embed_size,
    nhead=4,
    nhid=256,
    nlayers=2).to(device)
#TODO: investigate error? 

learning_rate = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = torch.nn.CrossEntropyLoss(ignore_index=pad_id)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 1.0, gamma=0.95)
#TODO: read more about schedular

/Users/slacksa/repos/chatGnT/src/chatGnT/models/transformer.py:17: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = TransformerEncoder(encoder_layers, nlayers)


In [97]:
best_val_loss = float("inf")
epochs = 12  # number of epochs
best_model = None

for epoch in range(1, epochs + 1):
    epoch_start_time = time.time()

    train.train(model, dataloader, device, pad_id, optimizer, criterion, epoch, 6)
    val_loss = evaluate.evaluate(model, dataloader, device, pad_id, criterion)

    print('-' * 89)
    print(
        f'Epoch {epoch} | Val Loss: {val_loss:.4f} | '
        f'Time {(time.time() - epoch_start_time)} | Val PPL: {math.exp(val_loss):.2f}')
        #TODO: currently not retruning training loss - should I? When look at each?
    print('-' * 89)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model = model

    scheduler.step()  # adjusts learning rate
    #TODO: read more about this?


Epoch 1 | Batch 0 | LR 0.001000 | Loss 1.0729 | PPL 2.92 | Time 0.02s
Epoch 1 | Batch 6 | LR 0.001000 | Loss 6.3376 | PPL 565.41 | Time 0.13s
Epoch 1 | Batch 12 | LR 0.001000 | Loss 6.2002 | PPL 492.84 | Time 0.13s
Epoch 1 | Batch 18 | LR 0.001000 | Loss 6.0660 | PPL 430.96 | Time 0.13s
-----------------------------------------------------------------------------------------
Epoch 1 | Val Loss: 5.8960 | Time 0.5170490741729736 | Val PPL: 363.57
-----------------------------------------------------------------------------------------
Epoch 2 | Batch 0 | LR 0.000950 | Loss 0.9948 | PPL 2.70 | Time 0.02s
Epoch 2 | Batch 6 | LR 0.000950 | Loss 5.9124 | PPL 369.61 | Time 0.13s
Epoch 2 | Batch 12 | LR 0.000950 | Loss 5.8094 | PPL 333.43 | Time 0.13s
Epoch 2 | Batch 18 | LR 0.000950 | Loss 5.6890 | PPL 295.61 | Time 0.13s
-----------------------------------------------------------------------------------------
Epoch 2 | Val Loss: 5.4681 | Time 0.5104451179504395 | Val PPL: 237.00
------------

## **Evaluate with Test Data**

In [98]:
#TODO: for now, use same data for test but should have separate test set
test_data = dataloader

test_loss = evaluate.evaluate(best_model, test_data, device, pad_id, criterion)
print('=' * 89)
print(f'Test Loss: {test_loss:5.2f} | Test PPL: {math.exp(test_loss):8.2f}')
print('=' * 89)

Test Loss:  3.11 | Test PPL:    22.33


## **Run with Test User Input**

In [99]:
#TODO: do need to do general string cleaning... Absinthe vs. absinthe
# <ingred>Absinthe</ingred>': 59
input = "Absinthe"

# tokenize
#TODO: think a different function would be better? Or just need function to handle all
# formats of user input to this format?
# input: df (pd.DataFrame): Must have columns ['amount', 'unit', 'ingred']
#TODO: for now just manually making token
input_mod = "<ingred>Absinthe</ingred>"

In [105]:
#TODO: how to embed?
#TODO: consider adding unknown ingredient token for things not in vocab?
#TODO: this is the encoding, not the embedding?

predict.predict(best_model, device, pad_id, vocab, inv_vocab, input_mod)




[59]


['<ingred>Absinthe</ingred>',
 '<amt>1</amt>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>',
 '<sep>']

In [107]:
predict.predict_groups(best_model, device, pad_id, vocab, inv_vocab, input_mod)

['<ingred>Absinthe</ingred>',
 '<sep>',
 '<ingred>Licorice root</ingred>',
 '<amt>1</amt>',
 '<unit>nan</unit>',
 '<amt>1</amt>',
 '<amt>1</amt>',
 '<amt>1</amt>',
 '<amt>2</amt>',
 '<amt>1</amt>',
 '<unit>nan</unit>',
 '<ingred>Advocaat</ingred>',
 '<ingred>Orange juice</ingred>',
 '<unit>tsp</unit>',
 '<amt>1 1/2</amt>',
 '<sep>',
 '<unit>nan</unit>',
 '<sep>']